In [5]:
from pyspark.sql import functions as F, types as T
import time
from spark_session import get_spark

In [3]:
spark = get_spark("udfs")
df = spark.range(3_000_000).withColumn("price", (F.col("id") % 1000) + 0.99)

In [4]:
def band(price):
    if price < 50:
        return "low"
    return "mid" if price < 200 else "high"

In [7]:
band_udf = F.udf(band, T.StringType())
builtin = F.when(F.col("price") < 50, "low").when(F.col("price") < 200, "mid").otherwise("high")

for name, expr in [("python udf", band_udf("price")), ("built-in", builtin)]:
    t = time.perf_counter()
    counts = df.withColumn("band", expr).groupBy("band").count().collect()
    print(f"{name}: {time.perf_counter() - t:.2f}s", sorted((r.band, r["count"]) for r in counts))
    

python udf: 11.49s [('high', 2400000), ('low', 150000), ('mid', 450000)]
built-in: 0.46s [('high', 2400000), ('low', 150000), ('mid', 450000)]


In [8]:
df.withColumn("band", band_udf("price")).explain()

== Physical Plan ==
*(2) Project [id#0L, price#1, pythonUDF0#40 AS band#39]
+- BatchEvalPython [band(price#1)#38], [pythonUDF0#40]
   +- *(1) Project [id#0L, (cast((id#0L % 1000) as double) + 0.99) AS price#1]
      +- *(1) Range (0, 3000000, step=1, splits=8)




In [9]:
@F.udf(returnType=T.StringType())
def initials(name):
    return "".join(w[0] for w in name.split()) if name else None

d:\Learn PySpark\.venv\Lib\site-packages\pyspark\sql\udf.py:120: RuntimeWarning: Arrow optimization failed to enable because PyArrow or Pandas is not installed. Falling back to a non-Arrow-optimized UDF.
  warnings.warn(


In [10]:
people = spark.createDataFrame([("Ada Lovelace",), ("Alan Turing", ), (None, )], "name STRING")
people.withColumn("initials", initials("name")).show()

+------------+--------+
|        name|initials|
+------------+--------+
|Ada Lovelace|      AL|
| Alan Turing|      AT|
|        NULL|    NULL|
+------------+--------+



In [11]:
spark.stop()